### `network_access_events`
VPN/network system. Represents external egress — the second link in an exfiltration chain (large internal download → external transfer).

| Field | Type | Notes |
|---|---|---|
| event_id | UUID (PK) | |
| system_identifier | string | |
| access_timestamp | timestamp | |
| source_ip | string | |
| destination_ip | string | |
| destination_system | string | Internal system name, or external label (e.g. "external_cloud_storage", "personal_email", "unknown_external") |
| bytes_transferred | float | |

Frequency: Human 0-10. Non-human: Supervised 0-6, Semi-autonomous 0-4, Fully-autonomous 0-2

Destination: 90% internal (auth_system, file_storage_system, admin_system, internal_network) / 10% external, split external_cloud_storage 60% / personal_email 25% / unknown_external 15%

bytes_transferred: internal uniform(1, 50), external uniform(50, 100)

Timing: same human/non-human pattern as the other three tables

In [0]:
pip install Faker

In [0]:
pip install pycountry

In [0]:
import uuid
import datetime
import random
import pycountry
import pytz
import zoneinfo
from faker import Faker
from pyspark.sql.functions import col, collect_set

In [0]:
# List of 44 european countries
europe_countries = [
    "Albania", "Andorra", "Austria", "Belarus", "Belgium", 
    "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", 
    "Denmark", "Estonia", "Finland", "France", "Germany", 
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", 
    "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", 
    "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia", 
    "Norway", "Poland", "Portugal", "Romania", "Russia", 
    "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", 
    "Sweden", "Switzerland", "Ukraine", "United Kingdom"
]

# System specs
system_list = ["internal", "external"]
system_weights = [0.9, 0.1]

# Internal specs
internal_list = ["auth_system", "file_storage_system", "admin_system", "internal_network"]
internal_weights = [0.1, 0.3, 0.2, 0.4]

# External specs
external_list = ["external_cloud_storage", "personal_email", "unknown_external"]
external_weights = [0.6, 0.25, 0.15]

fake = Faker()

In [0]:
entity_df = spark.read.table("entity_risk_platform.seed_data.entities").select("entity_id", "entity_type", "tier", "home_country")

network_identifier_df = spark.read.table("entity_risk_platform.seed_data.entity_system_identifiers").filter(col("system_name") == "network_access")

network_entity_df = network_identifier_df.join(entity_df, "entity_id")

# network_entity_df.show()

network_entities = [row.asDict() for row in network_entity_df.collect()]

# print(network_entities)

In [0]:
europe_timzones = {}

for ec in europe_countries:
    country = pycountry.countries.search_fuzzy(ec)[0]
    country_code = country.alpha_2
    country_timezone = pytz.country_timezones.get(country_code)[0]
    europe_timzones[ec] = country_timezone

# print(europe_timzones)

In [0]:
# chosen_entity = random.choice(network_entities)
# print(chosen_entity)

In [0]:
def generate_network_event_count(entity):
    event_count = 0
    if(entity["entity_type"] == "human"):
        event_count = random.randrange(0, 11)
    elif(entity["entity_type"] in ["service_account", "agent"]):
        if(entity["tier"] == "Fully-autonomous"):
            event_count = random.randrange(0, 3)
        elif(entity["tier"] == "Semi-autonomous"):
            event_count = random.randrange(0, 5)
        elif(entity["tier"] == "Supervised"):
            event_count = random.randrange(0, 7)

    return event_count

In [0]:
def generate_timestamp(date, timezone, min_sec, max_sec):
    random_date = datetime.datetime.combine(date.date(), datetime.time.min) + datetime.timedelta(seconds=random.randrange(min_sec, max_sec))
    local_date = random_date.replace(tzinfo=zoneinfo.ZoneInfo(timezone))
    utc_date = local_date.astimezone(zoneinfo.ZoneInfo("UTC"))
    return utc_date

# print(generate_timestamp(datetime.datetime.now(), chosen_entity["home_country"], 0, 3600))

In [0]:
def generate_network_timestamp(date, entity):
    if(entity["entity_type"] == "human"):
        # 8 AM
        min_seconds = 8 * 3600 
        # 6 PM
        max_seconds = 18 * 3600
        return generate_timestamp(date, europe_timzones[entity["home_country"]], min_seconds, max_seconds)
    else:
        # 12 AM
        min_seconds = 0
        # 12 PM
        max_seconds = 24 * 3600
        return generate_timestamp(date, "UTC", min_seconds, max_seconds)

In [0]:
def generate_network_download():
    system = random.choices(system_list, weights=system_weights)[0]
    if(system == "internal"):
        destination = random.choices(internal_list, weights=internal_weights)[0]
    else:
        destination = random.choices(external_list, weights=external_weights)[0]
    if(system == "internal"):
        volume = random.uniform(1, 50)
    else:
        volume = random.uniform(50, 100)
    return destination, volume

In [0]:
target_date = datetime.datetime.now()

In [0]:
network_event_list = []

In [0]:
def generate_network_events(date, entities):
    events = []
    for chosen_entity in entities:
        network_events = generate_network_event_count(chosen_entity)
        for _ in range(network_events):
            destination, volume = generate_network_download()
            events.append({
                "event_id": str(uuid.uuid4()),
                "system_identifier": chosen_entity["system_identifier"],
                "access_timestamp": generate_network_timestamp(date, chosen_entity),
                "source_ip": fake.ipv4(),
                "destination_ip": fake.ipv4(),
                "destination_system": destination,
                "bytes_transferred": volume
            })
    return events

network_event_list = generate_network_events(target_date, network_entities)
print(network_event_list)